# 19 — Director package: surfaces, tiers, clusters, tables (spec v1.7 on the curated block; v1.6 value-first structure)

**VERSION (study plan v0.17):** every surface, tier, cluster and star rebuilds on the curated-EFG-block re-solve (`config.Y2Y_VERSION = "v3"`, `runs_v3/`, records in `spec/v3/`). The package built on the artifact block is moved to `director_package/_superseded_v2_artifact_block/` on the first v3 run (supersede, never delete). Adds the adequacy-forced column from the necessity test (E19, notebook 18c) and the adequacy-pin caption rule for clusters ≥ 50% forced.

**Acts follow this analysis:** Act 0 = where the values are (value maps + convergence, the prologue), Act 1 = the core, Act 2 = the scenario tiers, Act 3 = the opportunity landscape (the measured gap). Votes = the 12 DESIGN formulations (study plan v0.15; `spec/manifest_v2.csv` role column from 13b).

Zero solves; runs after **18** has completed the guarded sweeps (14/14). The package votes with the **12 elicited positions** (the two crossed diagnostic hybrids are excluded — near-duplicate votes for S1/S3; the paper's registered F14 lives in 13). Everything here is the
GUARDED semantics (per-block floors; spec v0.14 applied headline), with the unguarded band carried
only for the T-D2 side-by-side. Writes `analyses/y2y/director_package/`:

- `geotiffs/` — `F_guarded.tif`, `F_unguarded.tif`, `f_guarded_<formulation>.tif` ×14,
  `union_membership_guarded.tif`, `act_tiers_guarded.tif`, `cluster_labels.npz`, `clusters.gpkg`
- `tables/` — `T-D1_cluster_register.csv`, `T-D2_bands.csv` + `T-D2_acts.csv`, `T-D3_scenarios.csv`,
  `tier_achievement.csv` (+ reference), `T-D5_protected_baseline.csv`, `T-D4_ecoregions.csv` (needs `input_data/ecoregions/`; pending otherwise),
  `pooling_check.csv`, `cluster_sensitivity.csv`, `cluster_register_all.csv`, `picks.csv`, `E17_shifts.csv`
- `summary.json` — headline numbers consumed by **20**

Pre-stated procedure (spec, unchanged): threshold ≥0.70 → closing r=1 → 8-connected components →
min 100 km² → scenario clusters minus the Act-1 core (overlap reported) → sensitivity at 0.60/0.80.
**Deck picks are COMPLEXES**: kept components within 75 km of each other (single linkage) are grouped for
presentation (Ethan 2026-09-04); components remain the analytic units in `cluster_register_all.csv`. Kernel `y2y-geo`.


In [1]:
import importlib, json, pathlib, shutil, sys
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

ALLOW_PARTIAL = False   # DEV ONLY: smoke-run on partial artifacts (outputs go to _smoke/)
assert config.EFG_SUBDIR == dc.VP.efg_subdir, f"config.EFG_SUBDIR {config.EFG_SUBDIR!r} does not match VERSION {dc.VP.version}"
# supersede, never delete: a package built on another manifest version is archived before this one is written
if not ALLOW_PARTIAL and (dc.PKG / "summary.json").exists():
    _old = json.loads((dc.PKG / "summary.json").read_text()).get("version", "v1")   # pre-v0.17 packages carried no version key
    if _old != dc.VP.version:
        _arch = dc.PKG / f"_superseded_{_old}_artifact_block"; _arch.mkdir(exist_ok=True)
        for _item in list(dc.PKG.iterdir()):
            if not _item.name.startswith("_"):
                shutil.move(str(_item), str(_arch / _item.name))
        print(f"archived the {_old} package to {_arch.relative_to(ROOT)}")
PKG = dc.ensure_dirs(dc.PKG / "_smoke" if ALLOW_PARTIAL else dc.PKG)
GEO, TAB = PKG / "geotiffs", PKG / "tables"
SPEC = dc.SPEC
G = dc.grid()
MAN_ALL = pd.read_csv(dc.MANIFEST)
assert len(MAN_ALL) in (12, 14), len(MAN_ALL)
MAN = dc.package_manifest(MAN_ALL)          # 12 DESIGN formulations (v0.15: s1x/s3x diagnostic, non-voting -- paper and package alike)
print(f"VERSION {dc.VP.version}: {dc.MANIFEST.name}; runs {dc.RUNS.relative_to(ROOT)}; EFG block {config.EFG_SUBDIR} ({dc.N_EFG} features)")
L = dc.load_guarded(G, MAN, allow_partial=ALLOW_PARTIAL)
FORMS = L.forms
THR = dc.FREQ_THR
SUMMARY = dict(version=dc.VP.version, manifest=dc.MANIFEST.name, efg_block=config.EFG_SUBDIR, n_efg=dc.N_EFG,
               efg_target_rule=(str(MAN_ALL.efg_target_rule.iloc[0]) if "efg_target_rule" in MAN_ALL.columns else "flat"),
               n_formulations=len(FORMS), excluded=list(dc.PACKAGE_EXCLUDE), threshold=THR, floor_g=dc.FLOOR_G, min_km2=dc.MIN_KM2,
               partial=ALLOW_PARTIAL, missing=L.missing)

cert = pd.DataFrame(L.cert).T
cert["D_plain"] = pd.Series(L.D_plain); cert["D_guard"] = pd.Series(L.D_guard)
print("\nguarded sweeps (results_log-ready):")
print(cert.to_string(float_format=lambda v: f"{v:.3f}"))
print(f"total guarded solve time {cert.runtime_min.sum() / 60:.1f} h | duplicates {int(cert.dup.sum())} | "
      f"time-limited {int(cert.time_limited.sum())}")
SUMMARY["guard_runtime_h"] = float(cert.runtime_min.sum() / 60)
# guarded MAA spot-check (S0): instrument-robustness of the GUARDED f
for fid, fm in L.f_maa_guard.items():
    fg = L.f_guard[fid]
    corr = float(np.corrcoef(fg[G.disc], fm[G.disc])[0, 1])
    J = dc.jaccard((fg >= THR) & G.disc, (fm >= THR) & G.disc)
    print(f"\nguarded MAA spot-check {fid}: corr(f_guardMGA, f_guardMAA) {corr:.3f} | frequent km2 "
          f"{int((fg[G.disc] >= THR).sum()):,} vs {int((fm[G.disc] >= THR).sum()):,} | tier Jaccard {J:.3f}")
    SUMMARY["maa_guard_spotcheck"] = dict(formulation=fid, corr=corr, tier_jaccard=J,
                                          freq_km2_mga=int((fg[G.disc] >= THR).sum()),
                                          freq_km2_maa=int((fm[G.disc] >= THR).sum()))


VERSION v3.1: manifest_v3.1.csv; runs analyses/y2y/runs_v3.1; EFG block iucn_efg_v3 (20 features)


s0_ssp585_theta5       f_guard freq  42,733 km2 | plain     650 km2 | D 1.000 -> 0.804


s1_ssp585_theta5       f_guard freq  74,160 km2 | plain  23,106 km2 | D 0.887 -> 0.664


s2_ssp585_theta5       f_guard freq  30,509 km2 | plain      29 km2 | D 1.000 -> 0.853


s3_ssp585_theta5       f_guard freq   9,636 km2 | plain       0 km2 | D 1.000 -> 0.949


...

s1_ssp245_theta5       f_guard freq  80,802 km2 | plain  34,491 km2 | D 0.836 -> 0.628


s2_ssp245_theta5       f_guard freq  33,108 km2 | plain      63 km2 | D 1.000 -> 0.846


s3_ssp245_theta5       f_guard freq  24,617 km2 | plain      21 km2 | D 1.000 -> 0.885


s4_ssp245_theta3       f_guard freq  60,395 km2 | plain  22,135 km2 | D 0.917 -> 0.718


s5_ssp245_theta5       f_guard freq  46,329 km2 | plain       0 km2 | D 1.000 -> 0.782

guarded sweeps (results_log-ready):
                      n   dup  runtime_min  time_limited  D_plain  D_guard
s0_ssp585_theta5 50.000 0.000       54.679         0.000    1.000    0.804
s1_ssp585_theta5 50.000 0.000       52.595         0.000    0.887    0.664
s2_ssp585_theta5 50.000 0.000       56.030         0.000    1.000    0.853
s3_ssp585_theta5 50.000 0.000       60.158         0.000    1.000    0.949
s4_ssp585_theta3 50.000 0.000       62.232         0.000    0.927    0.763
s5_ssp585_theta5 50.000 0.000       65.388         0.000    1.000    0.809
s0_ssp245_theta5 50.000 0.000       49.319         0.000    0.991    0.761
s1_ssp245_theta5 50.000 0.000       46.945         0.000    0.836    0.628
s2_ssp245_theta5 50.000 0.000       51.257         0.000    1.000    0.846
s3_ssp245_theta5 50.000 0.000       51.269         0.000    1.000    0.885
s4_ssp245_theta3 50.000 0.000       55.533         

In [2]:
# ---- surfaces: guarded F (deliverable) + unguarded F (side-by-side) + union membership ----------
Fg = dc.ensemble(L.f_guard, FORMS)
Fp = dc.ensemble(L.f_plain, FORMS)
Ug = dc.union_membership(L, FORMS, guarded=True)
Up = dc.union_membership(L, FORMS, guarded=False)
dc.write_tif(G, Fg, GEO / "F_guarded.tif")
dc.write_tif(G, Fp, GEO / "F_unguarded.tif")
dc.write_tif(G, Ug, GEO / "union_membership_guarded.tif")
for fid in FORMS:
    dc.write_tif(G, L.f_guard[fid], GEO / f"f_guarded_{fid}.tif")
print(f"wrote {2 + len(FORMS)} surfaces + union membership to {GEO.relative_to(ROOT)}")

# Act 1 by climate future: the same hierarchical F over the 6 formulations at each refugia realization
LEVELS = {"585": [f for f in FORMS if "ssp585" in f], "245": [f for f in FORMS if "ssp245" in f]}
F_LEV = {lv: dc.ensemble(L.f_guard, fids) for lv, fids in LEVELS.items() if fids}
for lv, F in F_LEV.items():
    dc.write_tif(G, F, GEO / f"F_guarded_ssp{lv}.tif")
core12 = (Fg >= THR) & G.disc
SUMMARY["by_level"] = {}
for lv, F in F_LEV.items():
    c = (F >= THR) & G.disc
    SUMMARY["by_level"][lv] = dict(n=len(LEVELS[lv]), core_km2=int(c.sum()), jaccard_vs_12=dc.jaccard(c, core12),
                                   pct_of_12_inside=float(100 * (c & core12).sum() / max(core12.sum(), 1)))
if len(F_LEV) == 2:
    c5, c2 = [(F_LEV[l] >= THR) & G.disc for l in ("585", "245")]
    SUMMARY["by_level"]["jaccard_585_245"] = dc.jaccard(c5, c2)
    SUMMARY["by_level"]["union_km2"] = int((c5 | c2).sum()); SUMMARY["by_level"]["intersection_km2"] = int((c5 & c2).sum())
print("\nAct 1 by climate future (6 formulations each) vs the 12-formulation core "
      f"({int(core12.sum()):,} km2):\n" + json.dumps(SUMMARY["by_level"], indent=1, default=float))

TD2a = dc.band_table(G, {"unguarded": Fp, "guarded": Fg, **{f"SSP{lv} only": F for lv, F in F_LEV.items()}})
TD2a.to_csv(TAB / "T-D2_bands.csv", index=False)
print("\nT-D2 (bands, discretionary landscape):")
print(TD2a.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
SUMMARY["frequent_km2"] = dict(guarded=int(((Fg >= THR) & G.disc).sum()), unguarded=int(((Fp >= THR) & G.disc).sum()))
SUMMARY["always_km2"] = dict(guarded=int(((Fg >= 0.95) & G.disc).sum()), unguarded=int(((Fp >= 0.95) & G.disc).sum()))

# Act 3 context: how much of the discretionary landscape is ever in a band
SUMMARY["ever_in_band_pct"] = dict(guarded=float(100 * (Ug[G.disc] > 0).mean()), unguarded=float(100 * (Up[G.disc] > 0).mean()))
p10 = dc.RUNS / "s0_ssp585_theta5" / "mga_g10.tif"
if p10.exists():
    u10 = ec.read_selections(p10, G.pu).any(axis=0)
    SUMMARY["s0_g10_union_pct"] = float(100 * u10[G.disc].mean())
    print(f"\never in a band: guarded {SUMMARY['ever_in_band_pct']['guarded']:.1f}% | unguarded "
          f"{SUMMARY['ever_in_band_pct']['unguarded']:.1f}% of discretionary cells; S0 alone at g=10%: "
          f"{SUMMARY['s0_g10_union_pct']:.1f}% (Gate-2b probe)")
# E11 sentence (from the Gate-4 record): ordered formulation pairs mutually within each other's 5% band
D11 = pd.read_csv(dc.SPEC_REC / "E11_delta_matrix.csv", index_col=0).loc[FORMS, FORMS]     # the 12 package positions
off = ~np.eye(len(D11), dtype=bool)
SUMMARY["e11_pairs_in_band"] = [int((D11.values[off] <= 0.05 + 1e-9).sum()), int(off.sum())]
print(f"E11: {SUMMARY['e11_pairs_in_band'][0]}/{SUMMARY['e11_pairs_in_band'][1]} ordered pairs mutually near-optimal")


wrote 14 surfaces + union membership to analyses/y2y/director_package/geotiffs



Act 1 by climate future (6 formulations each) vs the 12-formulation core (29,194 km2):
{
 "585": {
  "n": 6,
  "core_km2": 31358,
  "jaccard_vs_12": 0.6650259850963786,
  "pct_of_12_inside": 82.84236486949374
 },
 "245": {
  "n": 6,
  "core_km2": 41083,
  "jaccard_vs_12": 0.5286913773601323,
  "pct_of_12_inside": 83.25340823456875
 },
 "jaccard_585_245": 0.36308213378492804,
 "union_km2": 53145,
 "intersection_km2": 19296
}

T-D2 (bands, discretionary landscape):
                    band  unguarded km2  unguarded %disc  guarded km2  guarded %disc  SSP585 only km2  SSP585 only %disc  SSP245 only km2  SSP245 only %disc
      never [0.00, 0.05)            829              0.1       257765           23.8           274647               25.4           300398               27.8
       rare [0.05, 0.30)        1019185             94.2       637924           59.0           616916               57.0           593954               54.9
conditional [0.30, 0.70)          61867              5.7    

In [3]:
# ---- decision (g): climate-level pooling check; Act tiers; T-D2 act accounting ------------------
POOL, rep = dc.pool_scenarios(G, L.f_guard, MAN, force=True)     # deck: one map per scenario (both futures pooled)
POOLp, _ = dc.pool_scenarios(G, L.f_plain, MAN, force=True)
rep.to_csv(TAB / "pooling_check.csv", index=False)
print("pooling check (frequent-tier Jaccard between climate levels; pool iff >= "
      f"{dc.POOL_JACCARD_MIN}):\n{rep.to_string(index=False, float_format=lambda v: f'{v:.3f}')}")
SUMMARY["pooling"] = rep.to_dict(orient="records")

def act_masks(Fx, POOLx, Ux):
    core = (Fx >= THR) & G.disc
    out = {"Act 1 core (F >= 0.70, all 12 design formulations)": core}
    claimed = core.copy()
    any_sc = np.zeros(G.n_pu, bool)
    for key, f in POOLx.items():
        sid = key.split("@")[0]
        tier = (f >= THR) & G.disc & ~core
        tag = "Act 2" if sid in dc.ACT2_SCENARIOS else "appendix"
        out[f"{tag} {key}: {dc.SCENARIO_LABEL[sid]} (frequent minus core)"] = tier
        if sid in dc.ACT2_SCENARIOS:
            any_sc |= tier
    out["Act 2 any named scenario (union)"] = any_sc
    claimed |= any_sc
    out["Act 3 opportunity (in >= 1 band, not above)"] = (Ux > 0) & G.disc & ~claimed
    out["never (no near-optimal plan selects it)"] = (Ux == 0) & G.disc
    return out

AG, AP = act_masks(Fg, POOL, Ug), act_masks(Fp, POOLp, Up)
if len(F_LEV) == 2:      # climate-conditional core: which refugia future(s) a core cell survives
    c5, c2 = [(F_LEV[l] >= THR) & G.disc for l in ("585", "245")]
    AG["Act 1 core — BOTH refugia futures (cell-level intersection)"] = c5 & c2
    AG["climate-conditional core — SSP585 future only"] = c5 & ~c2
    AG["climate-conditional core — SSP245 future only"] = c2 & ~c5
rows = []
for k in AG:
    kp = AP.get(k)
    rows.append({"tier": k, "guarded km2": int(AG[k].sum()), "guarded %disc": 100 * AG[k].sum() / G.n_disc,
                 "unguarded km2": int(kp.sum()) if kp is not None else np.nan,
                 "unguarded %disc": 100 * kp.sum() / G.n_disc if kp is not None else np.nan})
TD2b = pd.DataFrame(rows)
TD2b.to_csv(TAB / "T-D2_acts.csv", index=False)
print("\nT-D2 (act tiers):")
print(TD2b.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
# Act-2 ownership: for each scenario-specific cell, WHICH named scenario gives it f >= 0.70 (pooled f);
# cells frequent under two or more scenarios are their own class. Codes: 0 none, 1..4 = ACT2_SCENARIOS order, 5 = 2+
core_m = AG["Act 1 core (F >= 0.70, all 12 design formulations)"]
owner = np.zeros(G.n_pu, np.uint8); nq = np.zeros(G.n_pu, np.uint8); best = np.full(G.n_pu, -1.0, np.float32)
for i, sid in enumerate(dc.ACT2_SCENARIOS, 1):
    if sid not in POOL:
        continue
    q = (POOL[sid] >= THR) & G.disc & ~core_m
    nq[q] += 1
    take = q & (POOL[sid] > best)
    owner[take] = i; best[take] = POOL[sid][take]
owner[nq >= 2] = 5
dc.write_tif(G, owner, GEO / "act2_owner.tif", dtype="uint8", nodata=255)
own_rows = []
for i, sid in enumerate(dc.ACT2_SCENARIOS, 1):
    own_rows.append({"tier": f"Act 2 by scenario — {dc.SCENARIO_LABEL[sid]} only (highest f where 2+ qualify excluded)",
                     "guarded km2": int((owner == i).sum()), "guarded %disc": 100 * (owner == i).sum() / G.n_disc,
                     "unguarded km2": np.nan, "unguarded %disc": np.nan})
own_rows.append({"tier": "Act 2 by scenario — frequent under 2+ named scenarios", "guarded km2": int((owner == 5).sum()),
                 "guarded %disc": 100 * (owner == 5).sum() / G.n_disc, "unguarded km2": np.nan, "unguarded %disc": np.nan})
TD2b = pd.concat([TD2b, pd.DataFrame(own_rows)], ignore_index=True)
TD2b.to_csv(TAB / "T-D2_acts.csv", index=False)
SUMMARY["act2_owner_km2"] = {dc.SCENARIO_LABEL[s]: int((owner == i).sum()) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)} | {"2+ scenarios": int((owner == 5).sum())}
print("\nAct 2 by scenario (km2):", SUMMARY["act2_owner_km2"])

# coded tier raster for the Act-3 map: 0 never / 1 opportunity / 2 scenario-specific / 3 core
tiers = np.zeros(G.n_pu, np.uint8)
tiers[AG["Act 3 opportunity (in >= 1 band, not above)"]] = 1
tiers[AG["Act 2 any named scenario (union)"]] = 2
tiers[AG["Act 1 core (F >= 0.70, all 12 design formulations)"]] = 3
dc.write_tif(G, tiers, GEO / "act_tiers_guarded.tif", dtype="uint8", nodata=255)


pooling check (frequent-tier Jaccard between climate levels; pool iff >= 0.8):
scenario  levels  jaccard                                                   decision  freq_km2_585  freq_km2_245
      s0       2    0.390 POOLED (forced; rule would separate at Jaccard 0.39 < 0.8)         42733         53834
      s1       2    0.422 POOLED (forced; rule would separate at Jaccard 0.42 < 0.8)         74160         80802
      s2       2    0.468 POOLED (forced; rule would separate at Jaccard 0.47 < 0.8)         30509         33108
      s3       2    0.285 POOLED (forced; rule would separate at Jaccard 0.29 < 0.8)          9636         24617
      s4       2    0.552 POOLED (forced; rule would separate at Jaccard 0.55 < 0.8)         50544         60395
      s5       2    0.382 POOLED (forced; rule would separate at Jaccard 0.38 < 0.8)         39585         46329

T-D2 (act tiers):
                                                             tier  guarded km2  guarded %disc  unguarded km2  u


Act 2 by scenario (km2): {'Core-habitat-forward': 31308, 'Connectivity-forward': 8429, 'Biodiversity-forward': 271, 'Carbon-forward': 23805, '2+ scenarios': 6353}


PosixPath('/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/analyses/y2y/director_package/geotiffs/act_tiers_guarded.tif')

In [4]:
# ---- clustering (pre-stated procedure) + sensitivity + top-k picks ------------------------------
named = dc.named_areas(G)
core2d = dc.to_grid(G, (Fg >= THR) & G.disc, fill=False, dtype=bool)

lab1, reg1 = dc.clusters(G, Fg)
reg1.insert(0, "act", "Act 1"); reg1.insert(1, "key", "ensemble")
reg1["name"] = [dc.placeholder_name(G, lab1 == c, named) if k else "" for c, k in zip(reg1.cid, reg1.kept)]
sens = [dc.sensitivity(G, Fg).assign(act="Act 1", key="ensemble")]
print(f"Act 1: tier {int(((Fg >= THR) & G.disc).sum()):,} km2 -> {len(reg1)} components, "
      f"{int(reg1.kept.sum())} >= {dc.MIN_KM2} km2 ({reg1[reg1.kept].km2.sum():,.0f} km2)")

LABELS = {"act1": lab1}
reg1, cx1 = dc.group_complexes(G, lab1, reg1)
REGS = [reg1]
CX = {"ensemble": cx1}                      # complexes per surface (presentational grouping, 75 km single linkage)
PICKS = []
number = 0
def pick_row(number, act, key, c, names):
    return dict(number=number, act=act, key=key, cid=int(c.anchor_cid), cids=";".join(map(str, c.cids)), n_components=int(c.n),
                name=names.get(int(c.anchor_cid), ""), km2=float(c.km2), meanF=float(c.meanF), lat=float(c.lat), lon=float(c.lon))
names1 = dict(zip(reg1.cid.astype(int), reg1["name"]))
for _, c in cx1.head(dc.TOPK_ACT1).iterrows():
    number += 1
    PICKS.append(pick_row(number, "Act 1", "ensemble", c, names1))
print(f"Act 1 complexes (single linkage {dc.COMPLEX_LINK_KM} km): {len(cx1)} from {int(reg1.kept.sum())} components; "
      f"top {dc.TOPK_ACT1}: " + ", ".join(f"{int(c.km2):,} km2 ({int(c.n)} comp.)" for _, c in cx1.head(dc.TOPK_ACT1).iterrows()))
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid not in dc.ACT2_SCENARIOS:
        continue
    lab, reg = dc.clusters(G, f, subtract2d=core2d)
    reg.insert(0, "act", "Act 2"); reg.insert(1, "key", key)
    reg["name"] = [dc.placeholder_name(G, (lab == c) & ~core2d, named) if k else "" for c, k in zip(reg.cid, reg.kept)]
    reg, cxk = dc.group_complexes(G, lab, reg)
    LABELS[f"act2_{key}"] = lab
    REGS.append(reg); CX[key] = cxk
    sens.append(dc.sensitivity(G, f, subtract2d=core2d).assign(act="Act 2", key=key))
    kept = reg[reg.kept]
    print(f"Act 2 {key:<8} ({dc.SCENARIO_LABEL[sid]}): {len(reg)} components, {len(kept)} kept after core "
          f"subtraction ({kept.residual_km2.sum():,.0f} km2 residual; mean core overlap of kept "
          f"{kept.core_overlap_pct.mean() if len(kept) else 0:.0f}%) -> {len(cxk)} complexes")
    namesk = dict(zip(reg.cid.astype(int), reg["name"]))
    for _, c in cxk.head(dc.TOPK_ACT2).iterrows():
        number += 1
        PICKS.append(pick_row(number, "Act 2", key, c, namesk))
# Act 1 by climate future: clusters on each 6-formulation core (own numbering "585-1..", "245-1..")
for lv, F in F_LEV.items():
    labL, regL = dc.clusters(G, F)
    regL.insert(0, "act", f"Act 1 ({lv})"); regL.insert(1, "key", f"ensemble_{lv}")
    regL["name"] = [dc.placeholder_name(G, labL == c, named) if k else "" for c, k in zip(regL.cid, regL.kept)]
    regL, cxL = dc.group_complexes(G, labL, regL)
    LABELS[f"act1_{lv}"] = labL
    REGS.append(regL); CX[f"ensemble_{lv}"] = cxL
    sens.append(dc.sensitivity(G, F).assign(act=f"Act 1 ({lv})", key=f"ensemble_{lv}"))
    namesL = dict(zip(regL.cid.astype(int), regL["name"]))
    for j, (_, c) in enumerate(cxL.head(dc.TOPK_ACT1).iterrows(), 1):
        PICKS.append(pick_row(f"{lv}-{j}", f"Act 1 ({lv})", f"ensemble_{lv}", c, namesL))
    print(f"Act 1 ({lv}): tier {int(((F >= THR) & G.disc).sum()):,} km2 -> {int(regL.kept.sum())} clusters >= {dc.MIN_KM2} km2")
REG = pd.concat(REGS, ignore_index=True)
REG.to_csv(TAB / "cluster_register_all.csv", index=False)
# ---- deck picks, second presentational tier: REGIONAL clusters (single linkage at PICK_LINK_KM) numbered NORTH -> SOUTH;
# the core first (1..n), then each scenario's picks continue the numbering; by-future picks keep their own labels
RAW_PICKS = pd.DataFrame(PICKS); RAW_PICKS.to_csv(TAB / "picks_raw_complexes.csv", index=False)
grouped, number = [], 0
core_raw = RAW_PICKS[RAW_PICKS.act == "Act 1"]
gp = dc.group_picks(G, LABELS["act1"], core_raw)
gp = dc.absorb_complexes(G, LABELS["act1"], gp, cx1, link_km=dc.PICK_LINK_KM, reg=reg1, speck_km=dc.SPECK_LINK_KM)   # nearby complexes AND sub-100 km2 specks join the nearest regional cluster (M4.33 addenda i, k)
for _, r in gp.iterrows():
    number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
print(f"core picks: {len(core_raw)} complexes -> {len(gp)} regional clusters (single linkage {dc.PICK_LINK_KM} km), numbered north -> south: "
      + "; ".join(f"{i + 1} = {r.name} ({r.km2:,.0f} km2; from picks {r.members})" for i, r in enumerate(gp.itertuples())))
for key in POOL:
    sc_raw = RAW_PICKS[(RAW_PICKS.act == "Act 2") & (RAW_PICKS.key == key)]
    if not len(sc_raw):
        continue
    gs = dc.group_picks(G, LABELS[f"act2_{key}"], sc_raw)
    for _, r in gs.iterrows():
        number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
grouped += [dict(members=str(r["number"]), **{k: v for k, v in r.items()}) for r in PICKS if str(r["act"]).startswith("Act 1 (")]
PICKS = grouped
SENS = pd.concat(sens, ignore_index=True)
SENS.to_csv(TAB / "cluster_sensitivity.csv", index=False)
PICKS = pd.DataFrame(PICKS)
PICKS.to_csv(TAB / "picks.csv", index=False)
np.savez_compressed(GEO / "cluster_labels.npz", **LABELS)
print("\nsensitivity companion (threshold 0.60 / 0.70 / 0.80):")
print(SENS.to_string(index=False, formatters={"threshold": "{:.2f}".format}, float_format=lambda v: f"{v:,.0f}"))
print("\ndeck picks (presentational top-k; register ships in full):")
print(PICKS.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

# cluster polygons (topology-preserving 2 km simplification) -> one GeoPackage, a layer per surface
gp = GEO / "clusters.gpkg"
if gp.exists():
    gp.unlink()
for k, lab in LABELS.items():
    keyname = "ensemble" if k == "act1" else (f"ensemble_{k[5:]}" if k.startswith("act1_") else k.replace("act2_", ""))
    picked = set(int(x) for _, r in PICKS[PICKS.key == keyname].iterrows() for x in str(r.cids).split(";") if str(x).strip())
    regk = REG[(REG.key == keyname) & (REG.kept | REG.cid.isin(picked))]        # kept components + the specks a cluster absorbed
    if len(regk):
        v = dc.vectorize(G, lab, regk.cid.tolist())
        v = v.merge(regk[["cid", "name", "km2", "meanF"]], on="cid")
        v.to_file(gp, layer=k, driver="GPKG")
print(f"wrote {gp.relative_to(ROOT)} ({len(LABELS)} layers)")


Act 1: tier 29,194 km2 -> 535 components, 46 >= 100 km2 (29,724 km2)


Act 1 complexes (single linkage 25 km): 15 from 46 components; top 6: 11,591 km2 (11 comp.), 10,962 km2 (12 comp.), 2,173 km2 (2 comp.), 1,093 km2 (4 comp.), 952 km2 (2 comp.), 711 km2 (2 comp.)


Act 2 s1       (Core-habitat-forward): 490 components, 47 kept after core subtraction (38,276 km2 residual; mean core overlap of kept 27%) -> 16 complexes


Act 2 s2       (Connectivity-forward): 1452 components, 20 kept after core subtraction (6,584 km2 residual; mean core overlap of kept 36%) -> 13 complexes


Act 2 s3       (Biodiversity-forward): 374 components, 3 kept after core subtraction (794 km2 residual; mean core overlap of kept 74%) -> 3 complexes


...

Act 1 (585): tier 31,358 km2 -> 40 clusters >= 100 km2


Act 1 (245): tier 41,083 km2 -> 53 clusters >= 100 km2


core picks: 6 complexes -> 4 regional clusters (single linkage 75 km), numbered north -> south: 1 = SW of Tahltan - Sacred Headwaters (Klappan) (56.8°N 129.3°W) (11,490 km2; from picks 2+53specks); 2 = Purcell Wilderness Conservancy Park vicinity (50.1°N 116.4°W) (12,105 km2; from picks 1+75specks); 3 = N of Granby Park (49.9°N 118.7°W) (2,507 km2; from picks 5+cx8+cx10+cx7+33specks); 4 = NW of Frank Church River Of No Return Wilderness (45.2°N 115.6°W) (5,229 km2; from picks 3;4;6+cx11+cx12+cx14+cx9+38specks)



sensitivity companion (threshold 0.60 / 0.70 / 0.80):
threshold  tier_km2  n_components  n_kept  kept_km2  largest_km2         act          key
     0.60     46321           638      51    46,850        9,361       Act 1     ensemble
     0.70     29194           535      46    29,724        6,699       Act 1     ensemble
     0.80     16540           443      35    15,695        5,526       Act 1     ensemble
     0.60     89457           600      65    89,730       21,159       Act 2           s1
     0.70     66090           490      47    64,204       10,960       Act 2           s1
     0.80     48912           465      40    43,908        9,398       Act 2           s1
     0.60     43816          1813      42    33,754       10,941       Act 2           s2
     0.70     27811          1452      20    19,289        4,420       Act 2           s2
     0.80     17912          1162       7     4,594        2,944       Act 2           s2
     0.60     24962           497      17    

wrote analyses/y2y/director_package/geotiffs/clusters.gpkg (7 layers)


In [5]:
# ---- T-D1 cluster register (every kept cluster; the deck shows the picks) -----------------------
P = dc.block_percentiles(G)
DM = dc.driver_masks(G)
IP = dc.ipca_layer(G)
near_pa = ndimage.distance_transform_edt(~G.locked2d) <= 5     # within 5 km of an existing PA
# the necessity test (E19, notebook 18c): adequacy-forced cells (capture 1.0 in every member of every design formulation)
E19_TIF = dc.RUNS / "e19_forced.tif"
if E19_TIF.exists():
    with rasterio.open(E19_TIF) as _s: FORCED = _s.read(1)[G.pu]
    with rasterio.open(dc.RUNS / "e19_forced_class.tif") as _s: FORCED_CLS = _s.read(1)[G.pu]
    FORCED_NAMES = {int(k): v for k, v in json.loads((dc.SPEC_REC / "E19_forced_classes.json").read_text()).items()}
    print(f"E19 forced layer loaded: {int((FORCED == 2).sum()):,} km2 forced in all formulations, {int((FORCED >= 1).sum()):,} in >= 1")
else:
    FORCED = None; print("E19 forced layer absent (run 18b + 18c after the re-solve) -- adequacy columns will be NaN")
rows = []
KEY2LAB = {"ensemble": "act1", **{f"ensemble_{lv}": f"act1_{lv}" for lv in F_LEV}, **{k: f"act2_{k}" for k in POOL}}
KEY2ACT = {"ensemble": "Act 1", **{f"ensemble_{lv}": f"Act 1 ({lv})" for lv in F_LEV}, **{k: "Act 2" for k in POOL}}
VR = dc.ValueRatios(G, P)                        # consequences: mean value in the cluster / mean over allocatable land
CXROWS = []
picked_cids = set()
for _, r in PICKS[~PICKS.act.str.startswith("Act 1 (")].iterrows():          # the deck picks (regional clusters) first
    cids = [int(c) for c in str(r.cids).split(";")]; picked_cids |= {(r.key, c) for c in cids}
    CXROWS.append(dict(act=r.act, key=r.key, cid=int(r.cid), cids=cids, name=r["name"], kept=True))
for key, cx in CX.items():                                                       # then every other kept complex
    regk = REG[REG.key == key].set_index("cid")
    for _, c in cx.iterrows():
        if any((key, int(x)) in picked_cids for x in c.cids):
            continue
        CXROWS.append(dict(act=KEY2ACT[key], key=key, cid=int(c.anchor_cid), cids=list(c.cids), name=regk.loc[int(c.anchor_cid), "name"], kept=True))
for r in pd.DataFrame(CXROWS).itertuples():
    lab = LABELS[KEY2LAB[r.key]]
    m2 = np.isin(lab, r.cids)
    if r.act == "Act 2":
        m2 = m2 & ~core2d
    m1 = m2[G.pu]
    n = int(m1.sum())
    prof = dc.star_profile(P, m1)
    pick = PICKS[(PICKS.act == r.act) & (PICKS.key == r.key) & (PICKS.cid == r.cid)]
    n_freq = int(sum(float(L.f_guard[fid][m1].mean() >= THR) >= 0.5 for fid in FORMS))  # formulations where >=50% of cells are frequent
    row = dict(number=str(pick.number.iloc[0]) if len(pick) else "", name=r.name, act=r.act, n_components=len(r.cids),
               driving=r.key if r.act == "Act 2" else "all formulations", area_km2=n * G.cell_km2,
               mean_guarded_F=float(Fg[m1].mean()), min_guarded_F=float(Fg[m1].min()))
    row.update({f"pct_{a}": prof[a] for a in dc.STAR_AXES})
    row.update({f"ratio_{a}": v for a, v in VR.of(m1).items()})           # consequences: x times the average allocatable cell
    row["efg_classes_present"] = dc.efg_classes_present(P, m1)
    row.update({f"driver_{k}": 100 * float(v[m1].mean()) for k, v in DM.items()})
    if FORCED is not None:
        fa = 100 * float((FORCED[m1] == 2).mean()); fy = 100 * float((FORCED[m1] >= 1).mean())
        cls = FORCED_CLS[m1 & (FORCED >= 1)]
        pin = FORCED_NAMES.get(int(np.bincount(cls).argmax()), "") if cls.size else ""
        row.update(pct_adequacy_forced=fa, pct_forced_any_formulation=fy, adequacy_pin=bool(fa >= 50), adequacy_pin_class=pin)
    else:
        row.update(pct_adequacy_forced=np.nan, pct_forced_any_formulation=np.nan, adequacy_pin=False, adequacy_pin_class="")
    row.update(mean_lat=float(dc.latlon(G)[0][m1].mean()),
               pct_within_5km_of_PA=100 * float(near_pa[m2].mean()),
               pct_in_proposed_IPCA=100 * float(IP.mask2d[m2].mean()),
               n_formulations_frequent=n_freq)
    rows.append(row)
TD1 = pd.DataFrame(rows).sort_values(["act", "area_km2"], ascending=[True, False]).reset_index(drop=True)
TD1.insert(3, "act_v16", TD1.act.map(dc.ACT_DISPLAY).fillna(TD1.act))
TD1.insert(5, "driving_label", [dc.SCENARIO_LABEL[d.split("@")[0]] if d != "all formulations" else "all formulations" for d in TD1.driving])
TD1.to_csv(TAB / "T-D1_cluster_register.csv", index=False)
show = ["number", "name", "act", "n_components", "driving_label", "area_km2", "mean_guarded_F", "mean_lat", "pct_in_proposed_IPCA",
        "n_formulations_frequent", "pct_adequacy_forced", "adequacy_pin_class"] + [c for c in TD1.columns if c.startswith("driver_")]
if FORCED is not None:
    pins = TD1[TD1.adequacy_pin]
    print(f"ADEQUACY PINS (>= 50% of the cluster forced in every formulation): {len(pins)} of {len(TD1)} clusters -- "
          + (", ".join(f"{r['name']} ({r.adequacy_pin_class})" for _, r in pins.iterrows()) if len(pins) else "none"))
print(f"T-D1 (kept complexes = components within {dc.COMPLEX_LINK_KM} km, single linkage; % columns are shares of cells):")
print(TD1[show].to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
RAT = TD1[TD1.act.isin(["Act 1", "Act 2"]) & (TD1.number != "")][["number", "name", "act", "driving_label", "area_km2", "mean_guarded_F"] + [f"ratio_{a}" for a in dc.STAR_AXES]].copy()
RAT["number"] = RAT.number.astype(int); RAT = RAT.sort_values("number")
# reference rows (Ethan 2026-09-14): existing protected areas and the unprotected part of the proposed IPCAs, same denominator
IPCA_ADD = IP.mask2d[G.pu] & ~G.locked
REF = pd.DataFrame([dict(number=np.nan, name="Existing protected areas", act="reference", driving_label="", area_km2=int(G.locked.sum()) * G.cell_km2,
                         mean_guarded_F=np.nan, **{f"ratio_{a}": v for a, v in VR.of(G.locked).items()}),
                    dict(number=np.nan, name="Proposed IPCAs (unprotected part)", act="reference", driving_label="", area_km2=int(IPCA_ADD.sum()) * G.cell_km2,
                         mean_guarded_F=float(Fg[IPCA_ADD].mean()), **{f"ratio_{a}": v for a, v in VR.of(IPCA_ADD).items()})])
RAT = pd.concat([RAT, REF], ignore_index=True)
RAT.to_csv(TAB / "T-D7_consequences.csv", index=False)
print(f"IPCA footprint on PU {int(IP.mask2d[G.pu].sum()):,} km2, of which {int((IP.mask2d[G.pu] & G.locked).sum()):,} already protected")
print("\nT-D7 consequences (mean value in the cluster / mean over allocatable land):")
print(RAT.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
print("\nNOTE clusters are discretionary by construction (PA overlap = 0), so 'pct_within_5km_of_PA' reports "
      "adjacency instead; cluster-IPCA overlap is INDEPENDENT CONVERGENCE (proposals are not locked in).")


driver masks: m_soc theta-tail 51,698 cells | refugia densest (same area) | spike 2,546 | rare-attainable EFG footprint 886,424 cells (70% of PU) from 17/20 EFGs | rarest-EFG footprint 7,504 cells (0.6%) from 2 EFGs


E19 forced layer loaded: 0 km2 forced in all formulations, 0 in >= 1


ADEQUACY PINS (>= 50% of the cluster forced in every formulation): 0 of 88 clusters -- none
T-D1 (kept complexes = components within 25 km, single linkage; % columns are shares of cells):
number                                                                             name         act  n_components        driving_label  area_km2  mean_guarded_F  mean_lat  pct_in_proposed_IPCA  n_formulations_frequent  pct_adequacy_forced adequacy_pin_class  driver_m_soc theta-tail  driver_connectivity spike (top 0.2%)  driver_refugia densest (area-matched to the m_soc tail)  driver_rare-attainable EFG footprint  driver_rarest-EFG footprint (rare in the extent+250 km window: <= 1% of the window)
     2                    Purcell Wilderness Conservancy Park vicinity (50.1°N 116.4°W)       Act 1            86     all formulations  12,105.0             0.8      50.6                   7.5                       10                  0.0                                         3.0                             

In [6]:
# ---- T-D3 scenario summary (from the frozen T1 record + anchors) ---------------------------------
cap = pd.read_csv(dc.SPEC_REC / "T1_anchor_captures.csv", index_col=0)
tail = pd.read_csv(dc.SPEC_REC / "T1_tail_capture.csv", index_col=0)
lat, _ = dc.latlon(G)
rows = []
for _, r in MAN.iterrows():
    fid = r.formulation_id
    if fid not in FORMS:
        continue
    row = dict(formulation=fid, scenario=dc.SCENARIO_LABEL[r.scenario_id], climate=r.climate_level.replace("_2071_2100", ""),
               value_statement=dc.SCENARIO_STATEMENT[r.scenario_id])
    for b, feats in config.BLOCKS.items():
        row[f"capture_{b}"] = float(cap.loc[fid, feats].mean())
    row["tail_m_soc"] = float(tail.loc[fid, "irrecoverable_carbon_m_soc"])
    row["tail_biomass"] = float(tail.loc[fid, "irrecoverable_carbon_biomass"])
    row["anchor_mean_lat"] = float(lat[L.anchors[fid] & G.disc].mean())
    row["frequent_km2_guarded"] = int((L.f_guard[fid][G.disc] >= THR).sum())
    row["frequent_km2_unguarded"] = int((L.f_plain[fid][G.disc] >= THR).sum())
    row["D_unguarded"], row["D_guarded"] = L.D_plain.get(fid, np.nan), L.D_guard.get(fid, np.nan)
    rows.append(row)
TD3 = pd.DataFrame(rows)
TD3.to_csv(TAB / "T-D3_scenarios.csv", index=False)
print("T-D3 (block captures = mean captured fraction of the block's features; tails = theta-tail mass capture):")
print(TD3.drop(columns="value_statement").to_string(index=False, float_format=lambda v: f"{v:,.3f}"))


T-D3 (block captures = mean captured fraction of the block's features; tails = theta-tail mass capture):
     formulation                       scenario climate  capture_core_habitat  capture_connectivity  capture_carbon  capture_biodiversity  tail_m_soc  tail_biomass  anchor_mean_lat  frequent_km2_guarded  frequent_km2_unguarded  D_unguarded  D_guarded
s0_ssp585_theta5                       Balanced  ssp585                 0.474                 0.318           0.352                 0.328       0.410         0.502           52.926                 42733                     650        1.000      0.804
s1_ssp585_theta5           Core-habitat-forward  ssp585                 0.504                 0.297           0.318                 0.319       0.362         0.300           52.570                 74160                   23106        0.887      0.664
s2_ssp585_theta5           Connectivity-forward  ssp585                 0.425                 0.363           0.329                 0.305     

In [7]:
# ---- v1.3 additions: tier-achievement (zero-solve) + T-D4 tier area by ecoregion ---------------
core1 = AG["Act 1 core (F >= 0.70, all 12 design formulations)"]
sc1 = AG["Act 2 any named scenario (union)"]
opp1 = AG["Act 3 opportunity (in >= 1 band, not above)"]
CUM = {"existing PAs": G.locked, "+ Act 1 core": G.locked | core1,
       "+ Act 2 scenario tiers": G.locked | core1 | sc1, "+ Act 3 opportunity": G.locked | core1 | sc1 | opp1}
TA = dc.tier_achievement(G, CUM)
# anchor-level reference: block captures of every anchor (T1 record) -> S0 + min/max across formulations
ref = TD3[[c for c in TD3.columns if c.startswith("capture_")]].rename(columns=lambda c: c.replace("capture_", ""))
ref.index = TD3.formulation
TA_ref = pd.DataFrame({"s0": ref.loc["s0_ssp585_theta5"] if "s0_ssp585_theta5" in ref.index else ref.iloc[0],
                       "anchor_min": ref.min(), "anchor_max": ref.max()})
TA.to_csv(TAB / "tier_achievement.csv", index=False); TA_ref.to_csv(TAB / "tier_achievement_reference.csv")
piv = TA[TA.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(list(CUM))
print("tier achievement (block capture, cumulative tiers incl. locked PAs):")
print(piv.to_string(float_format=lambda v: f"{v:.3f}"))
print("anchor reference (S0 / min / max across formulations):")
print(TA_ref.T.to_string(float_format=lambda v: f"{v:.3f}"))
SUMMARY["tier_area_pct_disc"] = {k: float(100 * (m & G.disc).sum() / G.n_disc) for k, m in
                                 [("core", core1), ("scenario", sc1), ("opportunity", opp1)]}

cap = pd.read_csv(dc.SPEC_REC / "T1_anchor_captures.csv", index_col=0)
# T-D5 protected baseline: what the existing PA estate already banks of each value (M3.6 accounting:
# PAs are locked in, so banked amounts COUNT toward targets and the 30% budget INCLUDES PA area)
sc0 = json.loads((SPEC / "scenarios_v2.json").read_text())["S0_balanced"]["targets"]
pa_area_share = float(G.locked.sum() / G.n_pu)
rows = []
for f in lc.continuous_features():
    v = np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0)
    pa = float(v[G.locked].sum() / v.sum()); tgt = float(sc0.get(f, 1.0))
    s0cap = float(cap.loc["s0_ssp585_theta5", f]) if "s0_ssp585_theta5" in cap.index else np.nan
    rows.append(dict(value=f, pct_of_regional_total_in_PAs=100 * pa, S0_target=tgt,
                     pct_of_target_already_banked=100 * pa / tgt,
                     pct_still_needed_from_unprotected_land=100 * max(tgt - pa, 0),
                     enrichment_existing_PAs=pa / pa_area_share,                       # capture share / area share
                     enrichment_S0_new_half=(s0cap - pa) / (config.BUDGET_PCT - pa_area_share)))  # the optimizer's 15%
TD5 = pd.DataFrame(rows)
TD5.to_csv(TAB / "T-D5_protected_baseline.csv", index=False)
n_efg_pa = int(P.efg[:, G.locked].any(axis=1).sum())
SUMMARY["protected_baseline"] = dict(pa_km2=int(G.locked.sum()), pa_pct_of_region=float(100 * G.locked.sum() / G.n_pu),
                                     pa_pct_of_budget=float(100 * G.locked.sum() / (config.BUDGET_PCT * G.n_pu)),
                                     efg_present_in_PAs=n_efg_pa, banked_min=float(TD5.pct_of_regional_total_in_PAs.min()),
                                     banked_max=float(TD5.pct_of_regional_total_in_PAs.max()))
print("T-D5 protected baseline (existing PAs, locked in every plan; enrichment = capture share / area share):")
print(TD5.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
print(f"PA estate {G.locked.sum():,} km2 = {SUMMARY['protected_baseline']['pa_pct_of_region']:.1f}% of the region = "
      f"{SUMMARY['protected_baseline']['pa_pct_of_budget']:.1f}% of the 30% budget; {n_efg_pa}/40 EFG classes present inside PAs\n")

# T-D5b enrichment by scenario: per value, capture share / area share for (i) each scenario anchor's NEW
# half (anchor capture minus the PA-banked share, over the 15% the optimizer chose) and (ii) each
# scenario's guarded FREQUENT tier (f >= 0.70, discretionary) plus the guarded ensemble core -- what the
# package promises. Anchors from the frozen T1 record (all 14); tiers from the guarded sweeps present.
VALS = {f: np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0) for f in lc.continuous_features()}
banked = {f: float(v[G.locked].sum() / v.sum()) for f, v in VALS.items()}
def tier_enrich(mask1d):
    a = mask1d.sum() / G.n_pu
    return {f: float(v[mask1d].sum() / v.sum()) / a if a > 0 else np.nan for f, v in VALS.items()}
E5 = {}
for _, r in MAN.iterrows():
    fid = r.formulation_id
    lab = f"{dc.SCENARIO_LABEL[r.scenario_id].split(' (')[0]} {'585' if 'ssp585' in fid else '245'}"
    E5[f"anchor · {lab}"] = {f: (float(cap.loc[fid, f]) - banked[f]) / (config.BUDGET_PCT - pa_area_share) for f in VALS}
    if fid in FORMS:
        E5[f"frequent tier · {lab}"] = tier_enrich((L.f_guard[fid] >= THR) & G.disc)
E5["frequent tier · ENSEMBLE core"] = tier_enrich((Fg >= THR) & G.disc)
E5["existing PAs"] = {f: banked[f] / pa_area_share for f in VALS}
TD5b = pd.DataFrame(E5)
TD5b.index.name = "value"
TD5b.to_csv(TAB / "T-D5b_enrichment_by_scenario.csv")
print("T-D5b enrichment (capture share / area share) -- anchors' new half vs guarded frequent tiers:")
show = [c for c in TD5b.columns if c.startswith("existing") or "585" in c or "ENSEMBLE" in c]
print(TD5b[show].T.to_string(float_format=lambda v: f"{v:.2f}"))

ECO = dc.ecoregion_layer(G)
if ECO is None:
    print(f"\nT-D4 PENDING: no ecozone/ecoregion vector in {dc.ECOREGIONS_DIR.relative_to(ROOT)}/ -- drop one there "
          "(e.g. CEC North American Level II/III ecoregions, seamless US+Canada) and re-run this cell")
    SUMMARY["td4"] = "pending (no ecoregion layer)"
else:
    z = ECO.zones[G.pu]
    tiers1 = np.zeros(G.n_pu, np.uint8); tiers1[opp1] = 1; tiers1[sc1] = 2; tiers1[core1] = 3
    rows = []
    for _, zr in ECO.gdf.iterrows():
        inz = z == zr.zone_id
        if inz.sum() == 0:
            continue
        rows.append({"ecoregion": zr[ECO.name_field], "PU km2": int(inz.sum()), "protected km2": int((inz & G.locked).sum()),
                     "core km2": int((inz & (tiers1 == 3)).sum()), "scenario km2": int((inz & (tiers1 == 2)).sum()),
                     "opportunity km2": int((inz & (tiers1 == 1)).sum()), "never km2": int((inz & G.disc & (tiers1 == 0)).sum()),
                     "mean lat": float(dc.latlon(G)[0][inz].mean())})
    TD4 = pd.DataFrame(rows).sort_values("core km2", ascending=False)
    TD4.to_csv(TAB / "T-D4_ecoregions.csv", index=False)
    SUMMARY["td4"] = f"from {ECO.source} ({ECO.name_field})"
    print(f"\nT-D4 (tier area by {ECO.name_field}, {ECO.source}):"); print(TD4.to_string(index=False))


tier achievement (block capture, cumulative tiers incl. locked PAs):
block                   biodiversity  carbon  connectivity  core_habitat
tier                                                                    
existing PAs                   0.156   0.149         0.146         0.192
+ Act 1 core                   0.181   0.176         0.170         0.278
+ Act 2 scenario tiers         0.234   0.307         0.233         0.364
+ Act 3 opportunity            0.983   0.993         0.984         0.991
anchor reference (S0 / min / max across formulations):
            core_habitat  connectivity  carbon  biodiversity
s0                 0.474         0.318   0.352         0.328
anchor_min         0.407         0.292   0.318         0.305
anchor_max         0.504         0.363   0.471         0.351


T-D5 protected baseline (existing PAs, locked in every plan; enrichment = capture share / area share):
                       value  pct_of_regional_total_in_PAs  S0_target  pct_of_target_already_banked  pct_still_needed_from_unprotected_land  enrichment_existing_PAs  enrichment_S0_new_half
          human_modification                          15.5        1.0                          15.5                                    84.5                      1.0                     1.0
  transboundary_connectivity                          14.9        1.0                          14.9                                    85.1                      1.0                     1.4
           climate_corridors                          14.3        1.0                          14.3                                    85.7                      1.0                     0.9
   climate_type_macrorefugia                          19.2        1.0                          19.2                                    80.8  

T-D5b enrichment (capture share / area share) -- anchors' new half vs guarded frequent tiers:
value                                     human_modification  transboundary_connectivity  climate_corridors  climate_type_macrorefugia  irrecoverable_carbon_biomass  irrecoverable_carbon_m_soc  aoh_richness_mammals  aoh_richness_birds
anchor · Balanced 585                                   1.01                        1.37               0.92                       1.88                          1.66                        1.04                  1.11                1.18
frequent tier · Balanced 585                            1.01                        1.21               0.91                       3.47                          1.18                        1.03                  1.06                1.08
anchor · Core-habitat-forward 585                       1.01                        1.10               0.91                       2.08                          1.21                        1.04         

In [8]:
# ---- Act 0 (where the values are), Act 3 (the measured gap), the hinge cross-tab ----------------
# Value = the block percentile already used for the star axes, thresholded at the top 30% of the DISCRETIONARY
# landscape; representativeness votes as presence of any rare-EFG class (the <= 1%-footprint set -- the 36
# rare-attainable classes cover 79% of the region and would vote almost everywhere; reported below, not used);
# naturalness is a sixth map, "disclosed, not a driver", outside the 0-5 convergence count.
rare_key = [k for k in DM if k.startswith("rarest-EFG")][0]
V = dc.value_layers(G, P, DM[rare_key])
for t, m in V.masks.items():
    dc.write_tif(G, m.astype(np.uint8), GEO / f"value_top30_{t.replace(' ', '_')}.tif", dtype="uint8", nodata=255)
dc.write_tif(G, V.convergence, GEO / "value_convergence.tif", dtype="uint8", nodata=255)
tier_code = tiers                                     # 0 never / 1 opportunity / 2 scenario / 3 core (cell 3)
high = (V.convergence >= 1) & G.disc
gap = high & (tier_code <= 1)                         # Act 3: high-value land that no tier makes irreplaceable
dc.write_tif(G, np.where(gap, V.convergence, 0).astype(np.uint8), GEO / "value_gap.tif", dtype="uint8", nodata=255)
TD6 = dc.coverage_table(G, V, tier_code)
alt = DM["rare-attainable EFG footprint"] & G.disc    # the alternative representativeness vote, reported not used
TD6.loc[len(TD6)] = {"theme": "representativeness — 36 rare-attainable classes (NOT used)", "top-value footprint km2": int(alt.sum()),
                     "% of unprotected land": 100 * alt.sum() / G.n_disc,
                     **{f"% of footprint in {nm}": 100 * float((alt & (tier_code == code)).sum()) / max(alt.sum(), 1) for code, nm in dc.TIER_NAMES.items()}}
TD6.to_csv(TAB / "T-D6_value_coverage.csv", index=False)
print("T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):")
print(TD6.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
# share of each block's regional VALUE by (non-cumulative) tier -- the coverage statement per theme
VS = dc.tier_achievement(G, {"existing PAs": G.locked, "core": core1, "scenario tiers": sc1, "opportunity": opp1, "never": G.disc & (tier_code == 0)})
VSp = VS[VS.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(["existing PAs", "core", "scenario tiers", "opportunity", "never"])
VSp.to_csv(TAB / "T-D6b_value_share_by_tier.csv")
print("\nshare of each block's regional value by tier (rows sum to 1 over the whole region):")
print(VSp.to_string(float_format=lambda v: f"{v:.3f}"))
XT = dc.crosstab(G, V.convergence, tier_code)
XT.to_csv(TAB / "hinge_crosstab.csv")
print("\nhinge cross-tab (km2): value-convergence count x reliability class:")
print(XT.to_string())
conv_km2 = {k: int(((V.convergence == k) & G.disc).sum()) for k in range(6)}
print(f"\nhigh-value land (>= 1 theme): {int(high.sum()):,} km2 = {100 * high.sum() / G.n_disc:.0f}% of unprotected land; "
      f"of it {int(gap.sum()):,} km2 ({100 * gap.sum() / max(high.sum(), 1):.0f}%) sits outside the core and the scenario tiers = Act 3")
# biodiversity: the finding is the product -- capture range over EVERY guarded plan (12 x 51) and the 12 anchors
bio = [np.nan_to_num(lc._read(config.HANDOFF_DIR / f"{f}.tif")[G.pu], nan=0.0) for f in config.BLOCKS["biodiversity"]]
bio_tot = [float(v.sum()) for v in bio]
caps = []
for fid in FORMS:
    Sg = np.vstack([L.anchors[fid][None, :], ec.read_selections(dc.RUNS / fid / "mga_guard_g05.tif", G.pu)])
    caps += [float(np.mean([v[row].sum() / t for v, t in zip(bio, bio_tot)])) for row in Sg]
    del Sg
caps = np.array(caps)
SUMMARY["biodiversity_plan_capture"] = dict(n_plans=int(caps.size), min=float(caps.min()), max=float(caps.max()), median=float(np.median(caps)))
print(f"biodiversity block capture over all {caps.size} guarded plans: {100 * caps.min():.1f}–{100 * caps.max():.1f}% (median {100 * np.median(caps):.1f}%)")
if FORCED is not None:
    conv_cont = sum(V.masks[t].astype(np.uint8) for t in ("core habitat", "connectivity", "biodiversity", "carbon"))
    def partition(m):
        n = max(int(m.sum()), 1); f = m & (FORCED == 2); mc = m & ~f & (conv_cont >= 2)
        return dict(km2=int(m.sum()), forced_pct=100 * f.sum() / n, multi_claim_pct=100 * mc.sum() / n, other_pct=100 * (m & ~f & ~mc).sum() / n)
    PARTS = {"core": core1, **{f"scenario tier: {dc.SCENARIO_LABEL[s]}": (owner == i) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)}, "opportunity": opp1}
    for _, r in PICKS.iterrows():
        lab = LABELS[KEY2LAB[r.key]]; m2 = np.isin(lab, [int(c) for c in str(r.cids).split(";")])
        if r.act == "Act 2": m2 = m2 & ~core2d
        PARTS[f"pick {r.number}: {r['name']}"] = m2[G.pu]
    E19P = pd.DataFrame([{"unit": k, **partition(m)} for k, m in PARTS.items()])
    E19P.to_csv(TAB / "E19_partition.csv", index=False)
    print("\nthe necessity test (E19) partition -- forced / multi-claim (>= 2 non-EFG themes top-30%) / other:")
    print(E19P.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    SUMMARY["e19"] = dict(core_forced_pct=float(E19P.iloc[0].forced_pct), n_pins=int(TD1.adequacy_pin.sum()), partition=E19P.to_dict(orient="records"))
rec_path = dc.SPEC_REC / "E11_recount_v015.json"
if rec_path.exists():
    SUMMARY["e11_recount_v015"] = json.loads(rec_path.read_text())
SUMMARY["value"] = dict(top=V.top, rare_efg_rule=rare_key, footprint_km2={t: int(m.sum()) for t, m in V.masks.items()},
                        convergence_km2=conv_km2, high_value_km2=int(high.sum()), gap_km2=int(gap.sum()),
                        gap_pct_of_high_value=float(100 * gap.sum() / max(high.sum(), 1)),
                        value_share_by_tier=VSp.to_dict(), coverage=TD6.to_dict(orient="records"), crosstab=XT.to_dict())


T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):
                                                     theme  top-value footprint km2  % of unprotected land  % of footprint in core  % of footprint in scenario tiers  % of footprint in opportunity  % of footprint in never
                                              core habitat                   324566                   30.0                     9.0                              13.3                           77.8                      0.0
                                              connectivity                   324567                   30.0                     2.2                               6.8                           91.1                      0.0
                                              biodiversity                   324641                   30.0                     4.6                               6.9                           88.5                      0.0
  


share of each block's regional value by tier (rows sum to 1 over the whole region):
block           biodiversity  carbon  connectivity  core_habitat
tier                                                            
existing PAs           0.156   0.149         0.146         0.192
core                   0.025   0.027         0.024         0.086
scenario tiers         0.053   0.131         0.063         0.087
opportunity            0.749   0.686         0.751         0.626
never                  0.017   0.007         0.016         0.009

hinge cross-tab (km2): value-convergence count x reliability class:
               core (F ≥ 0.70)  scenario tier  opportunity  never
0 of 5 themes                0            767       180469  22826
1 of 5 themes             2937          22869       469790   4428
2 of 5 themes            19374          40010       270405      9
3 of 5 themes             6451           6239        34307      0
4 of 5 themes              432            281          291   

biodiversity block capture over all 612 guarded plans: 29.0–35.1% (median 31.3%)



the necessity test (E19) partition -- forced / multi-claim (>= 2 non-EFG themes top-30%) / other:
                                                                         unit    km2  forced_pct  multi_claim_pct  other_pct
                                                                         core  29194         0.0             89.9       10.1
                                          scenario tier: Core-habitat-forward  31308         0.0             80.5       19.5
                                          scenario tier: Connectivity-forward   8429         0.0             56.6       43.4
                                          scenario tier: Biodiversity-forward    271         0.0             79.0       21.0
                                                scenario tier: Carbon-forward  23805         0.0             44.9       55.1
                                                                  opportunity 955262         0.0             31.6       68.4
         pick 1: SW of Tah

In [9]:
# ---- E17 inputs for the one-pager + summary.json ------------------------------------------------
base_lat, E17 = dc.e17_shifts(G)
E17.to_csv(TAB / "E17_shifts.csv", index=False)
E17_BASIS = str(E17.basis.iloc[0]) if len(E17) else "none"
if dc.VP.version != "v1" and E17_BASIS.startswith("v1"):
    print("NOTE: E17 leave-one-theme-out arms not yet solved on this version -- run the E17-T3 cell in 18b; the one-pager will show the v1 (40-class) bars")
_, E17_V1 = dc.e17_shifts(G, version="v1")        # the 40-class record, kept beside the curated-block result
geo = pd.read_csv(dc.SPEC_REC / "e17_efg_geography.csv")
n_south = int((geo.south_share > 0.90).sum())
SUMMARY["e17"] = dict(base_lat=base_lat, n_efg_south=n_south, n_efg=len(geo), median_efg_lat=float(geo.mean_lat.median()),
                      basis=E17_BASIS, shifts=E17.to_dict(orient="records"),
                      v1_shifts={r.block_out: float(r.delta_lat) for r in E17_V1.itertuples()})
t2p = dc.SPEC_REC / "E19_t2_anchors.csv"          # the necessity test's leave-EFG-out anchors re-measure E17 on the curated block
if t2p.exists():
    t2 = pd.read_csv(t2p).set_index("formulation"); d = (t2.lat_without - t2.lat_with)
    SUMMARY["e17"]["efg_out_curated"] = dict(s0_585=float(d.get("s0_ssp585_theta5", np.nan)), min=float(d.min()), max=float(d.max()), n=int(len(d)))
    print(f"E17 on the curated block (E19 T2): removing the ecosystem block shifts the anchors +{d.min():.2f} to +{d.max():.2f}N (S0 {d.get('s0_ssp585_theta5', float('nan')):+.2f}) vs +2.11N on the 40-class block")
print(f"E17: S0 anchor mean latitude {base_lat:.2f}N; {n_south}/{len(geo)} EFGs >90% south of 53N; shifts:\n"
      f"{E17.to_string(index=False, float_format=lambda v: f'{v:+.2f}')}")
SUMMARY["forms"] = FORMS
SUMMARY["pool_keys"] = list(POOL)
(PKG / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nwrote {PKG.relative_to(ROOT)}/summary.json -- next: 20_figures.ipynb")


E17 on the curated block (E19 T2): removing the ecosystem block shifts the anchors +0.02 to +0.22N (S0 +0.14) vs +2.11N on the 40-class block
E17: S0 anchor mean latitude 52.93N; 9/20 EFGs >90% south of 53N; shifts:
   block_out  mean_lat  delta_lat  jaccard_vs_s0 basis
core_habitat    +53.58      +0.66          +0.60  v3.1
connectivity    +51.16      -1.76          +0.77  v3.1
biodiversity    +55.19      +2.26          +0.76  v3.1
      carbon    +52.02      -0.91          +0.87  v3.1
         efg    +53.07      +0.14          +0.96  v3.1

wrote analyses/y2y/director_package/summary.json -- next: 20_figures.ipynb
